## REFERÊNCIA: https://chatgpt.com/share/684f8c22-27e4-8012-94e0-0868bcb3c438

In [ ]:
!pip install pandas spacy scikit-learn matplotlib seaborn wordcloud
!python -m spacy download en_core_web_sm


In [ ]:
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud

nlp = spacy.load("en_core_web_sm")


In [ ]:
# Substitua pelo caminho correto do seu Watson Studio (ou local)
df = pd.read_csv('/dataset/emails.csv') 

df.head()


In [ ]:
def preprocess_spacy(text):
    if pd.isnull(text):
        return ""
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
    return " ".join(tokens)

# Suponha que a coluna seja "Body" ou "message"
df["clean_text"] = df["Body"].apply(preprocess_spacy)


In [ ]:
# Seleciona uma amostra aleatória de 20 emails
amostra = df.sample(20, random_state=42)[["Body", "clean_text"]].copy()

# Aqui, rotule manualmente (0 = legítimo, 1 = engenharia social)
amostra["label"] = [0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(amostra["clean_text"])
y = amostra["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

modelo = MultinomialNB()
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print(classification_report(y_test, y_pred))


In [ ]:
texto_suspeito = " ".join(amostra[amostra["label"] == 1]["clean_text"])

wordcloud = WordCloud(width=800, height=400, background_color="white").generate(texto_suspeito)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Palavras comuns em textos de engenharia social")
plt.show()
